# RAG-over-PDF study tool — v1

This notebook is the "main" that wires the four pipeline stages together:

`parse → chunk → index → query`

Each stage lives in the `rag` package and writes a file to disk, so you can
inspect every intermediate artifact. v1 has **no OCR** and **no LLM** — the
output is a `context.txt` you paste into any chat model yourself.

**First time:** `pip install -r requirements.txt`. The embedding model
(~130 MB) downloads on first use and is cached afterwards.


## Setup
Put your slide PDFs in the `pdfs/` folder next to this notebook.

In [ ]:
from pathlib import Path
import rag

# --- paths -------------------------------------------------------------
PDF_DIR   = Path("G:\\Ca' Foscari\Semester 2")            # drop your .pdf slide decks here
WORK_DIR  = Path("artifacts")       # where intermediate files are written
WORK_DIR.mkdir(exist_ok=True)

RECORDS   = WORK_DIR / "records.jsonl"
CHUNKS    = WORK_DIR / "chunks.jsonl"
INDEX     = WORK_DIR / "index"          # -> index.npy + index.chunks.json
CONTEXT   = WORK_DIR / "context.txt"

# --- retrieval knobs ---------------------------------------------------
TOP_K  = 5     # how many independent chunks seed the result
WINDOW = 1     # how many neighbor slides (x-1, x+1) each seed drags in


## Stage 1 — Parse
PDFs → one record per slide (text + merged notes), saved as JSONL.

In [13]:
import json

records = rag.parse_directory(PDF_DIR)
rag.save_records(records, RECORDS)

n_docs  = len({r["doc_id"] for r in records})
n_empty = sum(1 for r in records if not r["has_text"])
print(f"Parsed {len(records)} slides across {n_docs} document(s).")
print(f"{n_empty} slide(s) had no extractable text (OCR candidates for later).")
print("\nExample record:")
print(json.dumps(records[0], indent=2, ensure_ascii=False)[:500])


Parsed 987 slides across 18 document(s).
1 slide(s) had no extractable text (OCR candidates for later).

Example record:
{
  "doc_id": "lecture0-intro",
  "slide_num": 1,
  "text": "Information Retrieval \nand Web Search\n(Introduction to the course)\nSalvatore Orlando\n1",
  "source_path": "G:\\Ca' Foscari\\Semester 2\\Information rtrival and web search\\lecture0-intro.pdf",
  "has_text": true,
  "has_notes": false
}


## Stage 2 — Chunk
v1 rule: one slide = one chunk. Empty (image-only) slides are dropped for now.

In [3]:
chunks = rag.chunk_records(records, drop_empty=True)
rag.save_records(chunks, CHUNKS)   # same JSONL writer works for chunks
print(f"{len(chunks)} chunks ready to index.")
print("Example chunk id:", chunks[0]["chunk_id"])


986 chunks ready to index.
Example chunk id: lecture0-intro::1


## Stage 3 — Embed & index
Embed every chunk with a local model and persist the matrix + chunks.

In [4]:
model = rag.load_model()                 # downloads on first run, then cached
index = rag.build_index(chunks, model)
rag.save_index(index, INDEX)
print(f"Indexed {len(index)} chunks with model '{index.model_name}'.")
print("Embedding matrix shape:", index.embeddings.shape)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\Ali\miniconda3\envs\rag-env\lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Ali\.cache\huggingface\hub\models--BAAI--bge-small-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/31 [00:00<?, ?it/s]

Indexed 986 chunks with model 'BAAI/bge-small-en-v1.5'.
Embedding matrix shape: (986, 384)


## Stage 4 — Query
Type your question below. The query is embedded, the top-k chunks retrieved,
their slide-neighbors pulled in, and the assembled context written to
`context.txt` (and printed). That file is your prompt — paste it into any chat
model along with your question.

In [5]:
question = "Explain how the chunking step relates to neighbor retrieval."

query_vec = rag.embed_query(question, model)
hits      = rag.retrieve(query_vec, index, k=TOP_K)

print("Top-k seeds:")
for chunk, score in hits:
    print(f"  {score:.3f}  {chunk['doc_id']} slide {chunk['slide_num']}")

expanded = rag.expand_neighbors(hits, index, window=WINDOW)
context  = rag.assemble_context(question, expanded)

CONTEXT.write_text(context, encoding="utf-8")
print(f"\nWrote {CONTEXT}  ({len(expanded)} chunks after neighbor expansion)\n")
print(context)


Top-k seeds:
  0.705  lecture4-indexconstruction slide 49
  0.692  lecture6-tfidf slide 3
  0.692  lecture6-tfidf (2) slide 3
  0.692  lecture6-tfidf (1) slide 3
  0.690  lecture4-indexconstruction slide 48

Wrote artifacts\context.txt  (13 chunks after neighbor expansion)

QUESTION:
Explain how the chunking step relates to neighbor retrieval.

CONTEXT:

[lecture4-indexconstruction - slide 47 | neighbor | score 0.690]
Other sorts of indexes
▪Positional indexes
▪Sort:  <termID, DocID, position>
▪Still sorting … just larger
▪Character n-gram indexes:
▪As text is parsed, enumerate n-grams
▪For each n-gram, need pointers to all dictionary terms 
containing it – the “postings”
▪… or postpone its construction once produced the 
complete lexicon
Sec. 4.5
for wildcards, spell checking, etc.
[NOTE] for wildcards, spell checking, etc.

[lecture4-indexconstruction - slide 48 | seed | score 0.690]
Sharding is a technique used in database systems and search engines 
to distribute/partition data (in

### One-call shortcut
Once the index exists you can skip the manual steps. `answer_to_file` runs
embed → retrieve → expand → write in one go.

In [12]:
rag.answer_to_file(
    "How to encode with golomb rice",
    index, model, CONTEXT, k=TOP_K, window=WINDOW,
)
print(CONTEXT.read_text(encoding="utf-8"))


QUESTION:
How to encode with golomb rice

CONTEXT:

[lecture5-compression - slide 52 | neighbor | score 0.806]
Alternative compression techniques
Sec. 5.3
52
•
Techniques
-
Unary encoding
-
Gamma encoding
-
Delta encoding
-
Variable byte encoding
-
Golomb encoding
-
Rice encoding
-
PforDelta encoding
-
Interpolative encoding
gap: 
1000
unary:    11111111…11111110 (999 ones)
gamma: 1111111110:111101000
delta: 
1110:010:111101000
vbe: 
00000111 11101000

[lecture5-compression - slide 53 | seed | score 0.806]
Other methods: Golomb-Rice
▪Choose an integer M = 2j , the modulus
▪Golomb-Rice is NOT parameter-free 
▪We want to encode the integer n > 0. 
▪Split n into two integer components, the quotient q(n) 
and the remainder r(n): 
▪q 𝑛=
𝑛−1
𝑀
▪r 𝑛= 𝑛−1 𝑚𝑜𝑑 𝑀
▪Encode n by writing: 
▪q 𝑛 in unary
▪followed by r 𝑛  as a binary number on j bits 
▪0 ≦𝑟𝑛< 𝑀  represented on 𝐥𝐨𝐠𝟐𝑴= 𝒋 bits  
▪Note that for representing M = 2j  we should need j+1 bits
53
Note we’re encoding  𝑛−1

[lecture5-compressio

### Reusing a saved index
In a fresh session you don't need to re-parse. Load the index and the model,
then query:

```python
import rag
model = rag.load_model()
index = rag.load_index("artifacts/index")
rag.answer_to_file("your question", index, model, "artifacts/context.txt")
```
